# Tutorial 9: Train NicheTrans on embryonic mouse brain data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_hd import *
from datasets.data_manager_MISAR_seq import ATAC_RNA_Seq

from utils.utils import *
from utils.utils_training_embryonic_mouse_brain import *
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_MISAR_seq.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.2, dropout_rate=0.1, n_source=3000, workers=4, knn_smooth=True, peak_threshold=0.05, hvg_gene=1500, adata_path='/mnt/datadisk0/Processed_DATA/2023_nm_MISAR_seq', max_epoch=20, stepsize=10, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = ATAC_RNA_Seq(peak_threshold=args.peak_threshold, hvg_gene=args.hvg_gene, adata_path=args.adata_path, RNA2ATAC=False, knn_smoothing=args.knn_smooth)
trainloader, testloader = embryonic_mouse_brain(args, dataset)

# create the model
source_dimension, target_dimension = len(dataset.source_panel), len(dataset.target_panel)
model = NicheTrans(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

------Calculating spatial graph...
The graph contains 17032 edges, 2129 cells.
8.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 14216 edges, 1777 cells.
8.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 15592 edges, 1949 cells.
8.0000 neighbors per cell on average.
source size 34323
target size 1500
=> Spatial atac-rna Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |   3906 spots,
  test     |   1949 spots,
  ------------------------------


### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.MSELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train_regression(model, criterion, optimizer, trainloader, device=device)
    if args.stepsize > 0: scheduler.step()
    ################

pearson = test_regression(model, testloader, device=device)
torch.save(model.state_dict(), 'NicheTrans_embryonic_mouse_brain_atac2rna.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20
Batch 122/122	 Loss 0.123858 (0.216793)
==> Epoch 2/20
Batch 122/122	 Loss 0.117310 (0.124550)
==> Epoch 3/20
Batch 122/122	 Loss 0.126276 (0.120614)
==> Epoch 4/20
Batch 122/122	 Loss 0.103554 (0.116117)
==> Epoch 5/20
Batch 122/122	 Loss 0.112059 (0.115291)
==> Epoch 6/20
Batch 122/122	 Loss 0.103836 (0.113120)
==> Epoch 7/20
Batch 122/122	 Loss 0.147580 (0.112256)
==> Epoch 8/20
Batch 122/122	 Loss 0.114087 (0.111573)
==> Epoch 9/20
Batch 122/122	 Loss 0.108666 (0.110894)
==> Epoch 10/20
Batch 122/122	 Loss 0.116975 (0.109064)
==> Epoch 11/20
Batch 122/122	 Loss 0.100464 (0.108420)
==> Epoch 12/20
Batch 122/122	 Loss 0.099968 (0.107033)
==> Epoch 13/20
Batch 122/122	 Loss 0.109982 (0.106071)
==> Epoch 14/20
Batch 122/122	 Loss 0.114797 (0.105053)
==> Epoch 15/20
Batch 122/122	 Loss 0.101198 (0.104918)
==> Epoch 16/20
Batch 122/122	 Loss 0.103603 (0.104757)
==> Epoch 17/20
Batch 122/122	 Loss 0.108817 (0.104962)
==> Epoch 18/20
Batch 122/122	 Loss 0.106803 (0.104527)
=